In [1]:
import pandas as pd
import numpy as np
import json
import src.utils.iv_helpers as iv_h
import src.utils.feature_eng as feat_eng
from src.utils.demeaning import demean_2FE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.linear_model import Lasso
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS
from sklearn.model_selection import GroupKFold
from pathlib import Path

# Loading Data
We load the panel and construct demeaned outcome and instrument matrix

In [2]:
IN = Path("../data/processed")

final = json.loads((IN / "final.json").read_text())
df = pd.read_parquet(IN / "bfs_data.parquet")

y = final["endog"]
Z = final["Z_selected"]

# Rebuilding multi-index
dfp = df.set_index([final["entity_level"], final["time_level"]]).sort_index()

# Demeaning using helper function
df_dm = demean_2FE(dfp[[y] + Z].dropna())
# Outcome vector
y_dm = df_dm[y]
# Regressor matrix
X_dm = df_dm[Z]

# Scaling for LASSO
Lasso is sensitive to scale so we standardize the instrument columns.

In [3]:
scaler = StandardScaler(with_mean=True, with_std=True)

X_std = scaler.fit_transform(X_dm.values)
dfX_std = pd.DataFrame(X_std, index=X_dm.index, columns=X_dm.columns)

# Checking if standardization worked (mean=0, std=1)
print(dfX_std.describe().loc[["mean","std"]].round().head())

      Z_temp_m2  Z_temp_m4  Z_temp_m5  Z_temp_m6  Z_temp_m7  Z_temp_m8  \
mean       -0.0       -0.0        0.0       -0.0        0.0       -0.0   
std         1.0        1.0        1.0        1.0        1.0        1.0   

      Z_prcp_m1  Z_prcp_m5  Z_prcp_m6  Z_prcp_m8  Z_prcp_m11  Z_prcp_m12  \
mean       -0.0        0.0        0.0        0.0         0.0        -0.0   
std         1.0        1.0        1.0        1.0         1.0         1.0   

      Z_prcp_m3  Z_temp_m3  
mean        0.0       -0.0  
std         1.0        1.0  


# Feature Engineering
We will transform the base instruments according to the following rules:

### Polynomials
+ $x = x^2$
+ $x = x^3$

### Absolute Value
+ $x = abs(x)$

### Hinges
+ $x = max[0, x-c_q]$
+ $x = max[0, c_q-x]$

In [4]:
X_temp = dfX_std.copy()

X_nonlinear = feat_eng.build_non_linear_feats(X_temp)

print(len(X_nonlinear.columns))
print(X_nonlinear.columns)

140
Index(['Z_temp_m2', 'Z_temp_m4', 'Z_temp_m5', 'Z_temp_m6', 'Z_temp_m7',
       'Z_temp_m8', 'Z_prcp_m1', 'Z_prcp_m5', 'Z_prcp_m6', 'Z_prcp_m8',
       ...
       'Z_prcp_m3__hinge_pos_q50', 'Z_prcp_m3__hinge_neg_q50',
       'Z_prcp_m3__hinge_pos_q75', 'Z_prcp_m3__hinge_neg_q75',
       'Z_temp_m3__hinge_pos_q25', 'Z_temp_m3__hinge_neg_q25',
       'Z_temp_m3__hinge_pos_q50', 'Z_temp_m3__hinge_neg_q50',
       'Z_temp_m3__hinge_pos_q75', 'Z_temp_m3__hinge_neg_q75'],
      dtype='object', length=140)


# Running LASSO Group K-fold
We choose the LASSO penalty α based on cross-validation that respects panel clustering.

In [5]:
X_vals = X_nonlinear.values
y_vals = y_dm.values

groups = X_nonlinear.index.get_level_values("entity_id").to_numpy()

gkf = GroupKFold(n_splits=5)

lasso_cv = LassoCV(
    cv=gkf.split(X_vals, y_vals, groups=groups),
    alphas=100,
    max_iter=20000,
    random_state=0,
)

lasso_cv.fit(X_vals, y_vals)

print("Chosen alpha:", lasso_cv.alpha_)
print("Nonzero coefs:", np.sum(lasso_cv.coef_ != 0))

Chosen alpha: 0.0002534347355137172
Nonzero coefs: 66


# Testing Over Subsamples For Stability
For robustness, we identify the instruments consistently picked across subsamples.

In [6]:
X_all = X_nonlinear.values
y_all = y_dm.values
feature_names = X_nonlinear.columns.to_numpy()

# Extracting alpha
alpha = float(lasso_cv.alpha_)

# Resampling entities
entities = X_nonlinear.index.get_level_values("entity_id").to_numpy()
unique_entities = np.unique(entities)

# No. sample runs
B = 200
# Fraction of entities per run
frac = 0.7
# Seeding
rng = np.random.default_rng(0)

sel_counts = np.zeros(X_all.shape[1], dtype=int)
coef_sums = np.zeros(X_all.shape[1], dtype=float)

for b in range(B):
    m = int(np.ceil(frac * len(unique_entities)))
    # Random sampling of entities
    sampled_ents = rng.choice(unique_entities, size=m, replace=False)

    mask = np.isin(entities, sampled_ents)
    X_b = X_all[mask]
    y_b = y_all[mask]

    # Fitting LASSO at fixed alpha
    model = Lasso(alpha=alpha, max_iter=20000, random_state=0)
    model.fit(X_b, y_b)

    nz = model.coef_ != 0
    sel_counts[nz] += 1
    coef_sums[nz] += model.coef_[nz]

# Selection frequencies + mean coef conditional on selection
sel_freq = sel_counts / B
mean_coef = np.zeros_like(coef_sums)
nonzero_mask = sel_counts > 0
mean_coef[nonzero_mask] = coef_sums[nonzero_mask] / sel_counts[nonzero_mask]

stable_cand = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "sel_freq": sel_freq,
            "mean_coef_if_selected": mean_coef,
        }
    )
    .sort_values(["sel_freq", "feature"], ascending=[False, True])
    .reset_index(drop=True)
)

stable_cand.head()

,feature,sel_freq,mean_coef_if_selected
0,Z_prcp_m12__hinge_pos_q25,1.000,-0.017425
1,Z_prcp_m5__cu,1.000,-0.002900
2,Z_temp_m2__cu,1.000,-0.002170
3,Z_temp_m7__cu,1.000,-0.001579
4,Z_temp_m4__cu,0.995,-0.001856


# Imposing Threshold On Stable Set
We keep the features that were picked above a certain threshold.

In [7]:
# Descriptive stats of candidate set
stable_cand.describe()

# Choosing median 0.7
THRESH = 0.7

# Filtering
stable_cand = stable_cand[stable_cand["sel_freq"] > THRESH]

print(stable_cand.shape)

(52, 3)


# Pruning stable set
Since too many variables remain, we prune the instrument set by imposing the following rule:

+ For a base feature, prefer its hinge transformation
+ Otherwise choose up to two of linear, cubic and square - but not square and cubic

In [8]:
stable = stable_cand.copy()
stable['feat_name'] = stable['feature'].apply(iv_h.feat_name)
stable['feat_trans'] = stable['feature'].apply(iv_h.feat_transformation)
stable['hinge_q'] = stable['feature'].apply(iv_h.hinge_quartile_search)

In [9]:
# Imposing stricter threshold
pooled = stable[stable["sel_freq"] >= 0.90].copy()

# Grouping by feature name
grouped = pooled.groupby('feat_name')

chosen_vars = []

for name, group in grouped:
    group = group.sort_values('sel_freq', ascending=False)

    # Hinge takes priority
    hinges = group[group['feat_trans'] == "hinge"]
    # Checking if hinges exist
    if len(hinges):
        # Choosing the best hinge
        chosen_vars.append(hinges.iloc[0]['feature'])
        continue

    # Otherwise check for other transformations and prefer linear/sq
    cand = group[group['feat_trans'].isin(['linear','square', 'cubic'])].copy()
    if len(cand):
        chosen = []
        for f in cand['feature']:
            # If 2 were chosen break the loop
            if len(chosen) >= 2: break
            # Avoid selecting both square and cubic
            if f.endswith('__cu') and any(x.endswith('__sq') for x in chosen):
                continue
            chosen.append(f)
        chosen_vars += chosen

Z_new = chosen_vars
print(Z_new, len(Z_new))

['Z_prcp_m1__hinge_neg_q25', 'Z_prcp_m11__cu', 'Z_prcp_m11__sq', 'Z_prcp_m12__hinge_pos_q25', 'Z_prcp_m3__cu', 'Z_prcp_m3__sq', 'Z_prcp_m5__cu', 'Z_prcp_m5__sq', 'Z_prcp_m6__cu', 'Z_prcp_m6__sq', 'Z_prcp_m8__hinge_neg_q75', 'Z_temp_m2__hinge_pos_q75', 'Z_temp_m3__cu', 'Z_temp_m3__sq', 'Z_temp_m4__hinge_pos_q25', 'Z_temp_m5__cu', 'Z_temp_m6__hinge_neg_q75', 'Z_temp_m7__cu', 'Z_temp_m7__sq', 'Z_temp_m8__cu'] 20


# Running FE on pruned set
We re-run the FE regression on the new pruned set of instruments. We then run a Wald test to verify the instruments have joint significance i.e. they have explanatory power.

In [10]:
# Creating new df with transformations
Z = Z_new
y = 'log_yield'

# Concatenating dfs
df_fe = pd.concat(
    [dfp[[y]], X_nonlinear[Z]],
    axis=1
).dropna()

model = PanelOLS(df_fe[y],
                 df_fe[Z],
                 entity_effects=True,
                 time_effects=True
                 ).fit(cov_type='clustered', cluster_entity=True)


# joint test that all Z coefficients are zero
wald = model.wald_test(formula=[f"{z} = 0" for z in Z])
print(wald)

Linear Equality Hypothesis Test
H0: Linear equality constraint is valid
Statistic: 339.5441
P-value: 0.0000
Distributed: chi2(20)


# Interpretation
We strongly reject null that instruments dont have jointly significant explanatory power at 0.01% confidence level.


# 2SLS
We run a two-stage least squares regression with the new instrument set to estimate its causal effect on area harvested in the following year. We then run an overidentification test.

In [11]:
# Saving list of instruments
Z_final = Z

X_nl = X_nonlinear.copy()

# Aligning index
dfp_aligned = dfp.loc[X_nl.index]

# Creating iv dataframe
df_iv = pd.concat([dfp_aligned, X_nl[Z_final]], axis=1).reset_index()

# Dropping duplicate cols
df_iv = df_iv.loc[:, ~df_iv.columns.duplicated()].copy()


In [12]:
Z = Z_final
endog = 'log_yield'
y = "log_area_lead1"

cols_needed = [y, endog] + Z

df_temp = df_iv.dropna(subset=cols_needed)

formula = f"{y} ~ 1 + [{endog} ~ {' + '.join(Z)}] + C(entity_id) + C(year)"

iv_res = IV2SLS.from_formula(
    formula,
    data=df_temp
).fit(cov_type="clustered", clusters=df_temp["entity_id"])

print(iv_res.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:         log_area_lead1   R-squared:                      0.9448
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9425
No. Observations:                2631   F-statistic:                  -5.5e+19
Date:                Thu, Jan 22 2026   P-value (F-stat)                1.0000
Time:                        15:37:45   Distribution:                chi2(105)
Cov. Estimator:             clustered                                         
                                                                              
                                    Parameter Estimates                                     
                          Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------------
Intercept                    11.321     0.6208     18.236     0.0000      10.104      12.

# Diagnostics Tests
## Wooldridge Score Test Of Overidentification
We reject the null that the model is not overidentified at 0.0001 confidence level. There is significant evidence that the model is overidentified.

In [13]:
print(iv_res.wooldridge_overid, '\n')

Wooldridge's score test of overidentification
H0: Model is not overidentified.
Statistic: 48.8657
P-value: 0.0002
Distributed: chi2(19) 



# Model Overidentified
We need to just identify it and pick the instrument with strongest first stage, reasonable SE and interpretable coefficient.

In [14]:
y = "log_area_lead1"
endog = "log_yield"
Z = Z_final

cols = [y, endog] + Z
df = df_iv.set_index(["entity_id","year"])[cols].dropna()

# Manually demeaning to avoid creating dummies
df_dm = demean_2FE(df)
y_dm = df_dm[y]
x_dm = df_dm[endog]

results = []

for z in Z:
    z_dm = df_dm[z]
    # Fitting each instrument individually
    r = IV2SLS(y_dm, None, x_dm, z_dm).fit(cov_type="clustered", clusters=df.index.get_level_values("entity_id"))

    fs = r.first_stage
    # Diagnostics for first stage
    diag = fs.diagnostics

    # Appending regression results to dictionary
    results.append({"Z": z,
                    "beta_log_yield": round(r.params["log_yield"], 3),
                    "se": round(r.std_errors["log_yield"],3),
                    "first_stage_F": round(diag.loc[endog, "f.stat"],3),
                    "first_stage_p": round(diag.loc[endog, "f.pval"],3),
                    "partial_R2": round(diag.loc[endog, "partial.rsquared"],3)})

# Sorting by standard error
results_df = pd.DataFrame(results).sort_values("se")
results_df.head()

,Z,beta_log_yield,se,first_stage_F,first_stage_p,partial_R2
17,Z_temp_m7__cu,-0.688,0.462,34.802,0.000,0.013
16,Z_temp_m6__hinge_neg_q75,-1.010,0.605,23.330,0.000,0.014
19,Z_temp_m8__cu,-0.507,0.729,12.542,0.000,0.007
6,Z_prcp_m5__cu,1.591,0.866,15.937,0.000,0.009
15,Z_temp_m5__cu,0.906,0.909,11.363,0.001,0.004


# Running IV With Strongest Instruments
We choose the three instruments from our instrument set with the highest F score in the first stage. We then run 2SLS with each of the instruments and record the diagnostics.

In [15]:
# Sorting by first stage F
Z_list = results_df.sort_values("first_stage_F", ascending=False)["Z"].head(3).tolist()
y = "log_area_lead1"
endog = "log_yield"

cols = [y, endog, "entity_id", "year"] + Z_list
df0 = df_iv[cols].dropna().copy()

rows = []
for z in Z_list:
    formula = f"{y} ~ 1 + C(entity_id) + C(year) + [{endog} ~ {z}]"
    r = IV2SLS.from_formula(formula, data=df0).fit(
        cov_type="clustered",
        clusters=df0["entity_id"]
    )
    diag = r.first_stage.diagnostics
    rows.append({
        "Z": z,
        "N": int(r.nobs),
        "beta": r.params[endog],
        "se": r.std_errors[endog],
        "F": diag.loc[endog, "f.stat"],
        "pF": diag.loc[endog, "f.pval"],
        "partial_R2": diag.loc[endog, "partial.rsquared"],
    })

df_iv_res = pd.DataFrame(rows).sort_values("F", ascending=False)

df_iv_res

,Z,N,beta,se,F,pF,partial_R2
0,Z_temp_m7__cu,2631,-0.697058,0.420028,34.674065,3.897900e-09,0.013478
1,Z_temp_m6__hinge_neg_q75,2631,-1.033206,0.556810,23.365087,1.339884e-06,0.014514
2,Z_prcp_m5__cu,2631,1.564490,0.849260,16.335516,5.306014e-05,0.009408


# Testing Exclusion Restriction
We check how sensitive the instruments are by adding nearby weather controls which could capture direct effects e.g. Z_temp_m7__cu gets Z_temp_m8__x. We then re-estimate 2SLS.

In [16]:
temp_months = [c for c in df_iv.columns if c.startswith("Z_temp_m")]
prcp_months = [c for c in df_iv.columns if c.startswith("Z_prcp_m")]

In [17]:
# Instrument is a temp month 7 cubed
iv_h.excl_test(df_iv, y="log_area_lead1", endog="log_yield",
          Z="Z_temp_m7__cu",
          controls=iv_h.controls_temp(7, temp_months=temp_months, prcp_months=prcp_months))

# Instrument is a temp month 6 cubed
iv_h.excl_test(df_iv, y="log_area_lead1", endog="log_yield",
          Z="Z_temp_m6__hinge_neg_q75",
          controls=iv_h.controls_temp(6, temp_months=temp_months, prcp_months=prcp_months))

# Instrument is a temp month 6 squared
iv_h.excl_test(df_iv, y="log_area_lead1", endog="log_yield",
          Z="Z_prcp_m5__cu",
          controls=iv_h.controls_prcp(5, temp_months=temp_months, prcp_months=prcp_months))


Exclusion restriction stress test for Z_temp_m7__cu
--------------------------------------
Baseline:  β = -0.697 (SE = 0.420)
Stress:    β = -0.460 (SE = 0.358)


Exclusion restriction stress test for Z_temp_m6__hinge_neg_q75
--------------------------------------
Baseline:  β = -1.033 (SE = 0.557)
Stress:    β = -0.804 (SE = 0.430)


Exclusion restriction stress test for Z_prcp_m5__cu
--------------------------------------
Baseline:  β = 1.564 (SE = 0.849)
Stress:    β = 1.316 (SE = 0.667)



# Final Regression
Our final 2SLS regression uses the strongest instrument -> $\text{Z\_temp\_m6\_\_hinge\_neg\_q75}$

In [18]:
y = "log_area_lead1"
endog = "log_yield"
z = "Z_temp_m6__hinge_neg_q75"

cols = [y, endog, z, "entity_id", "year"]
df_use = df_iv[cols].dropna().copy()

formula = f"{y} ~ 1 + C(entity_id) + C(year) + [{endog} ~ {z}]"
iv_final = IV2SLS.from_formula(formula, data=df_use).fit(
    cov_type="clustered",
    clusters=df_use["entity_id"]
)

print("beta:", iv_final.params[endog])
print("se:", iv_final.std_errors[endog])
print("p:", iv_final.pvalues[endog])
print(iv_final.first_stage.diagnostics)


beta: -1.0332062231609598
se: 0.5568097319370814
p: 0.06351313384781898
           rsquared  partial.rsquared  shea.rsquared     f.stat    f.pval  \
log_yield  0.840016          0.014514       0.014514  23.365087  0.000001   

            f.dist  
log_yield  chi2(1)  


# Conclusion
## Second Stage
+ $ β = -1.03 $
A 1% increase in the yield of the current year will lead to a 1.03% decrease in the area harvested (or equivalently planted) in the next year, conditional on year and entity (state-level) fixed effects. This is consistent with a supply adjustment.
+ $ p = 0.06 $
It is statistically significant at the 10% level of significance, but not at the 5% level. There is some evidence, but not strong.
## First stage
+ $\text{Partial} R^2 = 0.0145$
+ $p < 0.0001$
+ $\text{F-stat} = 23.37$
The F statistics is 23.37 which exceeds the threshold of 10 for a strong instrument. The partial R squared is small, explaining 1.45% of the variation in X, but is still enough to generate a strong first-stage.

## Economic Interpretation
The instrument captures **extreme negative shocks in June temperature**, which are below the 75 percentile threshold. This helps us isolate the variation which is exogenous to planting decision in the following year, except through it effect on yield in the current year. This holds if we assume that the exclusion restriction holds, that temperature shocks only affect planting decision through current yields.

## Analysis Conclusions
Over a broad set of linear and transformed instruments, June temperature shocks emerged endogenously as the most relevant and credible source of identifying variation. This suggests that information on the yield **mid-season** (6th month) plays the largest overall role on planting decision in the future.